# CNN-LSTM Multi-Output Model for Radar Spectrogram Classification

This notebook trains a CNN-LSTM model with two output heads:
- **Presence**: Binary classification (vibration vs no vibration)
- **Trend**: 3-class classification (constant, increasing, decreasing)

The trend output is only trained on samples where vibration is present.

## 1. Setup & GPU Verification

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU is available
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set memory growth to avoid OOM errors
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

## 2. Configuration

In [ ]:
import os
import glob
import time
import numpy as np
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# =============================================================================
# CONFIGURATION - MODIFY THESE PATHS AND PARAMETERS
# =============================================================================

# Data path (modify to your Google Drive path)
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/spectrograms-output/underground-pipe-vibration-2s"

# Spectrogram shape
TARGET_SHAPE = (163, 97)  # (frequency bins, time steps)
INPUT_SHAPE = (163, 97, 1)

# Model parameters
DROPOUT_RATE = 0.4
L2_REG = 1e-4

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.1

# Split configuration
USE_TIME_BASED_SPLIT = True

# Output configuration for multiple training runs
OUTPUT_BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"
NUM_TRAINING_RUNS = 10  # Number of times to train and record results

print("Configuration loaded!")
print(f"Data path: {DATA_PATH}")
print(f"Output base path: {OUTPUT_BASE_PATH}")
print(f"Number of training runs: {NUM_TRAINING_RUNS}")

## 3. Model Definition

In [ ]:
from tensorflow.keras import regularizers

def create_cnn_lstm_model(input_shape=(163, 97, 1), dropout_rate=0.4, l2_reg=1e-4):
    """
    Create a CNN-LSTM model with two output heads.

    Architecture:
    - CNN blocks with frequency-only downsampling (preserves temporal resolution)
    - Feature dimension reduction before LSTM
    - Bidirectional LSTM for temporal modeling
    - Two output heads: presence (binary) and trend (3-class)
    """
    inputs = tf.keras.Input(shape=input_shape, name="spectrogram")

    # ------------------------------------------------------------------
    # CNN feature extractor (frequency downsampling ONLY)
    # Preserves all 97 time steps for LSTM temporal modeling
    # ------------------------------------------------------------------

    # Conv Block 1 - wide frequency view
    x = tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(9, 3),
        strides=(2, 1),  # Downsample freq only
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="conv1",
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name="bn1")(x)
    x = tf.keras.layers.SpatialDropout2D(0.2, name="sd1")(x)

    # Conv Block 2 - mid-level patterns
    x = tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(7, 3),
        strides=(2, 1),
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="conv2",
    )(x)
    x = tf.keras.layers.BatchNormalization(name="bn2")(x)
    x = tf.keras.layers.SpatialDropout2D(0.25, name="sd2")(x)

    # Conv Block 3 - higher abstraction
    x = tf.keras.layers.Conv2D(
        filters=128,
        kernel_size=(5, 3),
        strides=(2, 1),
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="conv3",
    )(x)
    x = tf.keras.layers.BatchNormalization(name="bn3")(x)
    x = tf.keras.layers.SpatialDropout2D(0.3, name="sd3")(x)

    # ------------------------------------------------------------------
    # Reduce feature dimension before LSTM
    # ------------------------------------------------------------------
    x = tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(1, 1),
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="channel_reduce",
    )(x)

    # Permute and average over frequency dimension
    x = tf.keras.layers.Permute((2, 1, 3), name="permute_time_first")(x)
    x = tf.keras.layers.Lambda(
        lambda t: tf.reduce_mean(t, axis=2),
        name="freq_avg_pool"
    )(x)

    # ------------------------------------------------------------------
    # Temporal modeling with Bidirectional LSTM
    # ------------------------------------------------------------------
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2),
        name="bilstm1",
    )(x)

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(32, return_sequences=False, dropout=0.2),
        name="bilstm2",
    )(x)

    x = tf.keras.layers.Dropout(dropout_rate, name="lstm_dropout")(x)

    # Shared representation
    shared = tf.keras.layers.Dense(
        64, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="shared_dense"
    )(x)
    shared = tf.keras.layers.Dropout(0.3, name="shared_dropout")(shared)

    # ------------------------------------------------------------------
    # Output heads
    # ------------------------------------------------------------------

    # Presence head (binary: vibration vs no vibration)
    presence_output = tf.keras.layers.Dense(
        1, activation="sigmoid", name="presence"
    )(shared)

    # Trend head (3-class: constant, increasing, decreasing)
    trend_output = tf.keras.layers.Dense(
        3, activation="softmax", name="trend"
    )(shared)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "presence": presence_output,
            "trend": trend_output,
        },
        name="cnn_lstm_vibration",
    )

    return model

print("Model definition loaded!")

## 4. Custom Callbacks & Metrics

In [ ]:
class TimeLoggingCallback(tf.keras.callbacks.Callback):
    """Log epoch duration and total training time."""
    def on_train_begin(self, logs=None):
        self.epoch_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.time() - self.epoch_start_time
        self.epoch_times.append(epoch_time)
        print(f"\nEpoch {epoch + 1} completed in {epoch_time:.2f} seconds")

    def on_train_end(self, logs=None):
        total_time = time.time() - self.train_start_time
        avg_epoch_time = np.mean(self.epoch_times)
        print("=" * 60)
        print("TRAINING TIME SUMMARY")
        print("=" * 60)
        print(f"Total training time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
        print(f"Average epoch time: {avg_epoch_time:.2f}s")
        print(f"Fastest epoch: {min(self.epoch_times):.2f}s")
        print(f"Slowest epoch: {max(self.epoch_times):.2f}s")
        print("=" * 60)


class WarmUpLearningRateScheduler(tf.keras.callbacks.Callback):
    """Learning rate warmup scheduler."""
    def __init__(self, warmup_epochs, target_lr):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.target_lr = target_lr

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.target_lr * (epoch + 1) / self.warmup_epochs
            self.model.optimizer.learning_rate.assign(lr)
            print(f"\nWarmup LR: {lr:.6f}")


class F1Score(tf.keras.metrics.Metric):
    """Custom F1 Score metric for binary classification."""
    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()
        self.recall = tf.keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision.result()
        r = self.recall.result()
        return 2 * ((p * r) / (p + r + tf.keras.backend.epsilon()))

    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

print("Callbacks and metrics loaded!")

## 5. Data Loading Functions

In [ ]:
def augment_fn(inputs, targets):
    """
    SpecAugment-style augmentation for spectrograms.
    """
    image = inputs

    # Random horizontal flip (time reversal)
    if tf.random.uniform([]) > 0.5:
        image = tf.image.random_flip_left_right(image)

    # Time masking
    if tf.random.uniform([]) > 0.5:
        width = tf.shape(image)[1]
        mask_width = tf.random.uniform([], 1, 10, dtype=tf.int32)
        mask_start = tf.random.uniform([], 0, width - mask_width, dtype=tf.int32)

        mask = tf.concat([
            tf.ones([tf.shape(image)[0], mask_start, tf.shape(image)[2]]),
            tf.zeros([tf.shape(image)[0], mask_width, tf.shape(image)[2]]),
            tf.ones([tf.shape(image)[0], width - mask_start - mask_width, tf.shape(image)[2]])
        ], axis=1)
        image = image * mask

    # Frequency masking
    if tf.random.uniform([]) > 0.5:
        height = tf.shape(image)[0]
        mask_height = tf.random.uniform([], 1, 15, dtype=tf.int32)
        mask_start = tf.random.uniform([], 0, height - mask_height, dtype=tf.int32)

        mask = tf.concat([
            tf.ones([mask_start, tf.shape(image)[1], tf.shape(image)[2]]),
            tf.zeros([mask_height, tf.shape(image)[1], tf.shape(image)[2]]),
            tf.ones([height - mask_start - mask_height, tf.shape(image)[1], tf.shape(image)[2]])
        ], axis=0)
        image = image * mask

    return image, targets


def load_spectrograms_by_class(data_path, target_shape=(163, 97)):
    """
    Load spectrograms from the four category folders, sorted for time-based split.
    """
    data = {}
    mapping = {
        "no-vibration":          (0, 0),
        "full-vibration":        (1, 0),
        "increasing-vibration":  (1, 1),
        "decreasing-vibration":  (1, 2),
    }

    for folder, (pres, tr) in mapping.items():
        pattern = os.path.join(data_path, folder, "*.npy")
        files = sorted(glob.glob(pattern))
        print(f"Loading {len(files)} files from '{folder}' (sorted)...")

        spectrograms = []
        for f in files:
            spec = np.load(f)
            if spec.shape != target_shape:
                zoom_factors = (target_shape[0] / spec.shape[0], target_shape[1] / spec.shape[1])
                spec = zoom(spec, zoom_factors, order=1)
            spectrograms.append(spec)

        if len(spectrograms) > 0:
            X = np.array(spectrograms, dtype=np.float32)
            X = np.expand_dims(X, axis=-1)
            data[folder] = {
                'X': X,
                'presence': pres,
                'trend': tr,
            }
            print(f"  -> Loaded {len(X)} samples, shape {X.shape}")

    return data


def time_based_split_multiclass(data, train_ratio=0.7, val_ratio=0.15):
    """
    Split data chronologically for each class.
    Returns arrays and sample weights for trend masking.
    """
    test_ratio = 1.0 - train_ratio - val_ratio

    X_train, X_val, X_test = [], [], []
    y_pres_train, y_pres_val, y_pres_test = [], [], []
    y_trend_train, y_trend_val, y_trend_test = [], [], []
    trend_weight_train, trend_weight_val, trend_weight_test = [], [], []

    for folder, class_data in data.items():
        X = class_data['X']
        pres = class_data['presence']
        trend = class_data['trend']

        n = len(X)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        X_train.append(X[:train_end])
        X_val.append(X[train_end:val_end])
        X_test.append(X[val_end:])

        y_pres_train.extend([pres] * train_end)
        y_pres_val.extend([pres] * (val_end - train_end))
        y_pres_test.extend([pres] * (n - val_end))

        y_trend_train.extend([trend] * train_end)
        y_trend_val.extend([trend] * (val_end - train_end))
        y_trend_test.extend([trend] * (n - val_end))

        weight = 1.0 if pres == 1 else 0.0
        trend_weight_train.extend([weight] * train_end)
        trend_weight_val.extend([weight] * (val_end - train_end))
        trend_weight_test.extend([weight] * (n - val_end))

    X_train = np.concatenate(X_train)
    X_val = np.concatenate(X_val)
    X_test = np.concatenate(X_test)

    y_pres_train = np.array(y_pres_train, dtype=np.int32)
    y_pres_val = np.array(y_pres_val, dtype=np.int32)
    y_pres_test = np.array(y_pres_test, dtype=np.int32)

    y_trend_train = np.array(y_trend_train, dtype=np.int32)
    y_trend_val = np.array(y_trend_val, dtype=np.int32)
    y_trend_test = np.array(y_trend_test, dtype=np.int32)

    trend_weight_train = np.array(trend_weight_train, dtype=np.float32)
    trend_weight_val = np.array(trend_weight_val, dtype=np.float32)
    trend_weight_test = np.array(trend_weight_test, dtype=np.float32)

    print(f"\nTime-based split (chronological):")
    print(f"  Train: first {train_ratio*100:.0f}% of each class = {len(X_train)} samples")
    print(f"  Val:   next {val_ratio*100:.0f}% = {len(X_val)} samples")
    print(f"  Test:  last {test_ratio*100:.0f}% = {len(X_test)} samples")
    print(f"\nTrend samples (vibration only): Train={np.sum(trend_weight_train > 0):.0f}, Val={np.sum(trend_weight_val > 0):.0f}, Test={np.sum(trend_weight_test > 0):.0f}")

    return (X_train, X_val, X_test,
            y_pres_train, y_pres_val, y_pres_test,
            y_trend_train, y_trend_val, y_trend_test,
            trend_weight_train, trend_weight_val, trend_weight_test)


def normalize_spectrograms(X_train, X_val, X_test):
    """Normalize spectrograms using training set statistics."""
    mean = np.mean(X_train)
    std = np.std(X_train)

    print(f"\nNormalization stats: Mean={mean:.4f}, Std={std:.4f}")

    X_train_norm = (X_train - mean) / (std + 1e-8)
    X_val_norm = (X_val - mean) / (std + 1e-8)
    X_test_norm = (X_test - mean) / (std + 1e-8)

    return X_train_norm, X_val_norm, X_test_norm, mean, std

print("Data loading functions ready!")

## 6. Load Data

In [ ]:
# Check if data path exists
if not os.path.exists(DATA_PATH):
    print(f"ERROR: Data path not found: {DATA_PATH}")
    print("Please update DATA_PATH in the Configuration cell.")
else:
    print("Loading spectrograms...")
    load_start = time.time()
    data = load_spectrograms_by_class(DATA_PATH, target_shape=TARGET_SHAPE)
    print(f"\nData loading completed in {time.time() - load_start:.2f}s")

    if len(data) == 0:
        print("\nNo data found! Expected folders:")
        print("  - no-vibration/")
        print("  - full-vibration/")
        print("  - increasing-vibration/")
        print("  - decreasing-vibration/")

## 8. Split & Normalize Data

In [ ]:
# Time-based split
(X_train, X_val, X_test,
 y_pres_train, y_pres_val, y_pres_test,
 y_trend_train, y_trend_val, y_trend_test,
 trend_weight_train, trend_weight_val, trend_weight_test) = time_based_split_multiclass(data)

# Normalize
X_train, X_val, X_test, train_mean, train_std = normalize_spectrograms(X_train, X_val, X_test)

print("\nData ready for training!")

## 9. Create TensorFlow Datasets

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Training dataset with augmentation
train_dataset = tf.data.Dataset.from_tensor_slices((
    X_train,
    {"presence": y_pres_train, "trend": y_trend_train},
    {"presence": np.ones_like(y_pres_train, dtype=np.float32), "trend": trend_weight_train}
))
train_dataset = train_dataset.shuffle(buffer_size=len(X_train))
train_dataset = train_dataset.map(
    lambda x, y, w: (augment_fn(x, y)[0], y, w),
    num_parallel_calls=AUTOTUNE
)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(AUTOTUNE)

# Validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((
    X_val,
    {"presence": y_pres_val, "trend": y_trend_val},
    {"presence": np.ones_like(y_pres_val, dtype=np.float32), "trend": trend_weight_val}
))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Test dataset
test_dataset = tf.data.Dataset.from_tensor_slices((
    X_test,
    {"presence": y_pres_test, "trend": y_trend_test},
    {"presence": np.ones_like(y_pres_test, dtype=np.float32), "trend": trend_weight_test}
))
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Datasets created with trend masking for no-vibration samples!")

## 10. Build & Compile Model

In [ ]:
# Model will be created fresh for each training run in the loop below
# Just verify the model architecture here
print("Verifying CNN-LSTM model architecture...")
test_model = create_cnn_lstm_model(input_shape=INPUT_SHAPE, dropout_rate=DROPOUT_RATE, l2_reg=L2_REG)
test_model.summary()
del test_model  # Clean up
print("\nModel architecture verified! Training loop will create fresh models.")

## 11. Train Model

In [ ]:
# Run training loop for multiple iterations
all_run_results = []

for run_idx in range(NUM_TRAINING_RUNS):
    print("\n" + "=" * 60)
    print(f"TRAINING RUN {run_idx + 1} / {NUM_TRAINING_RUNS}")
    print("=" * 60)
    
    # Create output folder for this run
    run_folder = f"{OUTPUT_BASE_PATH}/model{run_idx}"
    os.makedirs(run_folder, exist_ok=True)
    model_save_path = f"{run_folder}/best_cnn_lstm.keras"
    
    # Create fresh model for this run
    print(f"\nCreating fresh CNN-LSTM model for run {run_idx}...")
    tf.keras.backend.clear_session()  # Clear memory from previous run
    model = create_cnn_lstm_model(input_shape=INPUT_SHAPE, dropout_rate=DROPOUT_RATE, l2_reg=L2_REG)
    
    # Compile model
    optimizer = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    losses = {
        "presence": tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        "trend": tf.keras.losses.SparseCategoricalCrossentropy(),
    }
    loss_weights = {"presence": 1.0, "trend": 1.0}
    metrics = {
        "presence": [
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            F1Score(name="f1_score"),
        ],
        "trend": [
            tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            F1ScoreSparse(name="f1_score", num_classes=3),
        ],
    }
    model.compile(optimizer=optimizer, loss=losses, loss_weights=loss_weights, metrics=metrics)
    
    # Callbacks for this run
    time_callback = TimeLoggingCallback()
    warmup_callback = WarmUpLearningRateScheduler(WARMUP_EPOCHS, LEARNING_RATE)
    
    callbacks = [
        time_callback,
        warmup_callback,
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            model_save_path, monitor="val_loss", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
    ]
    
    # Train
    train_start = time.time()
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=callbacks,
        verbose=1
    )
    training_time = time.time() - train_start
    print(f"\nTraining run {run_idx} completed in {training_time:.2f}s")
    
    # Evaluate on test set
    print(f"\nEvaluating run {run_idx} on test set...")
    test_results = model.evaluate(test_dataset, verbose=1)
    
    # Store results
    run_result = {
        "run_idx": run_idx,
        "training_time": training_time,
        "test_results": dict(zip(model.metrics_names, test_results)),
        "history": history.history,
    }
    all_run_results.append(run_result)
    
    # --- Generate and save confusion matrices ---
    predictions = model.predict(X_test, verbose=0)
    presence_proba = predictions["presence"]
    trend_proba = predictions["trend"]
    
    y_pred_presence = (presence_proba > 0.5).astype(int).flatten()
    y_pred_trend = np.argmax(trend_proba, axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    cm_presence = confusion_matrix(y_pres_test, y_pred_presence)
    sns.heatmap(cm_presence, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Vibration', 'Vibration'],
                yticklabels=['No Vibration', 'Vibration'], ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title(f'Confusion Matrix - Vibration Presence (Run {run_idx})')
    
    vibration_mask = y_pres_test == 1
    y_trend_test_filtered = y_trend_test[vibration_mask]
    y_pred_trend_filtered = y_pred_trend[vibration_mask]
    
    cm_trend = confusion_matrix(y_trend_test_filtered, y_pred_trend_filtered)
    sns.heatmap(cm_trend, annot=True, fmt='d', cmap='Greens',
                xticklabels=['Constant', 'Increasing', 'Decreasing'],
                yticklabels=['Constant', 'Increasing', 'Decreasing'], ax=axes[1])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title(f'Confusion Matrix - Vibration Trend (Run {run_idx})')
    
    plt.tight_layout()
    plt.savefig(f'{run_folder}/confusion_matrices.png', dpi=150)
    plt.show()
    
    # --- Generate and save training history plots ---
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    axes[0, 0].plot(history.history['presence_loss'], label='Train')
    axes[0, 0].plot(history.history['val_presence_loss'], label='Val')
    axes[0, 0].set_title('Presence Loss')
    axes[0, 0].legend()
    
    axes[0, 1].plot(history.history['presence_accuracy'], label='Train')
    axes[0, 1].plot(history.history['val_presence_accuracy'], label='Val')
    axes[0, 1].set_title('Presence Accuracy')
    axes[0, 1].legend()
    
    axes[0, 2].plot(history.history['presence_auc'], label='Train')
    axes[0, 2].plot(history.history['val_presence_auc'], label='Val')
    axes[0, 2].set_title('Presence AUC')
    axes[0, 2].legend()
    
    axes[1, 0].plot(history.history['trend_loss'], label='Train')
    axes[1, 0].plot(history.history['val_trend_loss'], label='Val')
    axes[1, 0].set_title('Trend Loss')
    axes[1, 0].legend()
    
    axes[1, 1].plot(history.history['trend_accuracy'], label='Train')
    axes[1, 1].plot(history.history['val_trend_accuracy'], label='Val')
    axes[1, 1].set_title('Trend Accuracy')
    axes[1, 1].legend()
    
    axes[1, 2].plot(history.history['loss'], label='Train')
    axes[1, 2].plot(history.history['val_loss'], label='Val')
    axes[1, 2].set_title('Total Loss')
    axes[1, 2].legend()
    
    for ax in axes.flat:
        ax.set_xlabel('Epoch')
    
    plt.tight_layout()
    plt.savefig(f'{run_folder}/training_history.png', dpi=150)
    plt.show()
    
    # Save normalization stats
    norm_stats_path = model_save_path.replace('.keras', '_norm_stats.npy')
    np.save(norm_stats_path, {"mean": train_mean, "std": train_std})
    
    print(f"\nRun {run_idx} saved to: {run_folder}")

print("\n" + "=" * 60)
print(f"ALL {NUM_TRAINING_RUNS} TRAINING RUNS COMPLETED!")
print("=" * 60)

## 12. Evaluate on Test Set

In [ ]:
# Summary of all training runs
print("=" * 60)
print("SUMMARY OF ALL TRAINING RUNS")
print("=" * 60)

for result in all_run_results:
    run_idx = result['run_idx']
    test_res = result['test_results']
    print(f"\nRun {run_idx}:")
    print(f"  Training time: {result['training_time']:.2f}s")
    for name, value in test_res.items():
        print(f"  {name}: {value:.4f}")

# Calculate average metrics
print("\n" + "=" * 60)
print("AVERAGE METRICS ACROSS ALL RUNS")
print("=" * 60)

metric_names = all_run_results[0]['test_results'].keys()
for metric in metric_names:
    values = [r['test_results'][metric] for r in all_run_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    print(f"{metric}: {mean_val:.4f} ± {std_val:.4f}")

## 13. Confusion Matrices

In [ ]:
# Confusion matrices are generated and saved in the training loop above
# Each run's confusion matrix is saved to model{idx}/confusion_matrices.png
print("Confusion matrices were saved during training runs.")
print("Check the following folders:")
for run_idx in range(NUM_TRAINING_RUNS):
    print(f"  - {OUTPUT_BASE_PATH}/model{run_idx}/confusion_matrices.png")

## 14. Training History Plots

In [ ]:
# Training history plots are generated and saved in the training loop above
# Each run's history is saved to model{idx}/training_history.png
print("Training history plots were saved during training runs.")
print("Check the following folders:")
for run_idx in range(NUM_TRAINING_RUNS):
    print(f"  - {OUTPUT_BASE_PATH}/model{run_idx}/training_history.png")

## 15. Save Model & Normalization Stats

In [ ]:
# All models and stats are saved in the training loop above
print("All models and normalization stats have been saved.")
print("\nSaved files:")
for run_idx in range(NUM_TRAINING_RUNS):
    folder = f"{OUTPUT_BASE_PATH}/model{run_idx}"
    print(f"\nRun {run_idx}:")
    print(f"  Model: {folder}/best_cnn_lstm.keras")
    print(f"  Norm stats: {folder}/best_cnn_lstm_norm_stats.npy")
    print(f"  Confusion matrices: {folder}/confusion_matrices.png")
    print(f"  Training history: {folder}/training_history.png")

print("\n" + "=" * 60)
print("ALL DONE!")
print("=" * 60)